In [1]:
# Import required python modules
import os
import numpy as np
import matplotlib.pyplot as plt
import healpy as hp
import pandas as pd
import sqlite3
from astropy.coordinates import SkyCoord
import astropy.units as u
import rubin_scheduler.scheduler.utils as sched_utils
from rubin_scheduler.utils import ddf_locations, ddf_locations_skycoord
from rubin_sim.data import get_baseline
import rubin_sim.maf as maf
from scipy.stats import binned_statistic

from IPython.display import display, Markdown

In [8]:
# v3.4 runs
basedir = '/sdf/group/rubin/web_data/sim-data/sims_featureScheduler_runs3.4'
accor_dir = os.path.join(basedir, '/ddf_acor')

In [10]:
# v3.4 runs

summaries = maf.get_metric_summaries(summary_source=os.path.join(basedir, 'maf/summary.h5'))
print(f"This summary h5 file contains information on {len(summaries.index)} simulations.")
print(summaries.index)

This summary h5 file contains information on 265 simulations.
Index(['all_long_v3.4_10yrs', 'baseline_v2.0_10yrs', 'baseline_v2.1_10yrs',
       'baseline_v2.2_10yrs', 'baseline_v3.0_10yrs', 'baseline_v3.2_10yrs',
       'baseline_v3.3_10yrs', 'baseline_v3.4_10yrs',
       'ddf_acor_sf10_lsf15_lsr10_v3.4_10yrs',
       'ddf_acor_sf10_lsf15_lsr20_v3.4_10yrs',
       ...
       'weather_cloudso1v3.4_10yrs', 'weather_cloudso20v3.4_10yrs',
       'weather_cloudso2v3.4_10yrs', 'weather_cloudso30v3.4_10yrs',
       'weather_cloudso31v3.4_10yrs', 'weather_cloudso35v3.4_10yrs',
       'weather_cloudso36v3.4_10yrs', 'weather_cloudso4v3.4_10yrs',
       'weather_cloudso6v3.4_10yrs', 'weather_cloudso8v3.4_10yrs'],
      dtype='object', name='run', length=265)


In [11]:
metric_subsets = maf.get_metric_subsets('metric_subsets.json')
msets = list(metric_subsets.groupby('metric subset').first().index)

m5_SRD_design = {'u': 23.90, 'g': 25.00, 'r': 24.70, 'i': 24.00, 'z': 23.30, 'y': 22.10}
m5_SRD_minimum = {'u': 23.40, 'g':24.60, 'r': 24.30, 'i': 23.60, 'z': 22.90, 'y': 21.70}

baselinerun = 'baseline_v3.4_10yrs'
baseline = baselinerun

ddf_accord = [r for r in summaries.index if 'ddf_acor' in r]

ddf_sets = [ddf_accord]
ddf_runs =  [r for r in ddf_accord]

outdir = 'tmp_fig'
try:
    os.mkdir(outdir)
except:
    pass

In [12]:
all_runs =  [baselinerun] + ddf_runs
all_runs

['baseline_v3.4_10yrs',
 'ddf_acor_sf10_lsf15_lsr10_v3.4_10yrs',
 'ddf_acor_sf10_lsf15_lsr20_v3.4_10yrs',
 'ddf_acor_sf10_lsf15_lsr50_v3.4_10yrs',
 'ddf_acor_sf10_lsf20_lsr10_v3.4_10yrs',
 'ddf_acor_sf10_lsf20_lsr20_v3.4_10yrs',
 'ddf_acor_sf10_lsf20_lsr50_v3.4_10yrs',
 'ddf_acor_sf10_lsf30_lsr10_v3.4_10yrs',
 'ddf_acor_sf10_lsf30_lsr20_v3.4_10yrs',
 'ddf_acor_sf10_lsf30_lsr50_v3.4_10yrs',
 'ddf_acor_sf10_lsf40_lsr10_v3.4_10yrs',
 'ddf_acor_sf10_lsf40_lsr20_v3.4_10yrs',
 'ddf_acor_sf10_lsf40_lsr50_v3.4_10yrs',
 'ddf_acor_sf15_lsf20_lsr10_v3.4_10yrs',
 'ddf_acor_sf15_lsf20_lsr20_v3.4_10yrs',
 'ddf_acor_sf15_lsf20_lsr50_v3.4_10yrs',
 'ddf_acor_sf15_lsf30_lsr10_v3.4_10yrs',
 'ddf_acor_sf15_lsf30_lsr20_v3.4_10yrs',
 'ddf_acor_sf15_lsf30_lsr50_v3.4_10yrs',
 'ddf_acor_sf15_lsf40_lsr10_v3.4_10yrs',
 'ddf_acor_sf15_lsf40_lsr20_v3.4_10yrs',
 'ddf_acor_sf15_lsf40_lsr50_v3.4_10yrs',
 'ddf_acor_sf20_lsf0_lsr0_v3.4_10yrs',
 'ddf_acor_sf20_lsf30_lsr10_v3.4_10yrs',
 'ddf_acor_sf20_lsf30_lsr20_v3.4_10

In [14]:
conn = sqlite3.connect(os.path.join(basedir, 'baseline', baselinerun + ".db"))
query = 'select night, moonPhase from observations where night<100'
vals = pd.read_sql(query, conn)
nstart = vals.query('moonPhase < 50').night.min()
print(nstart)

0


In [19]:
# Basic DDF information, number of visits from baseline simulation
ddfs = ddf_locations_skycoord()
nvis = {}
frac_seeing = {}
u_per_30 = {}
u_per_30 = {}
max_u_per_30 = {}
pair_count = {}
for run in all_runs:
    if run.startswith('baseline'):
        conn = sqlite3.connect(os.path.join(basedir, 'baseline', run + ".db"))
    else:
        conn = sqlite3.connect(os.path.join(basedir, 'ddf_acor', run + ".db"))
    nvis[run] = {}
    frac_seeing[run] = {}
    u_per_30[run] = {}
    max_u_per_30[run] = {}
    pair_count[run] = {}
    for i in ddfs:
        query = f"select count(*) from observations where scheduler_note like '%{i}%'"
        nvis[run][i] = int(pd.read_sql(query, conn).values[0][0])
        query = f"select seeingFwhmEff from observations where scheduler_note like '%{i}%' and filter == 'r'"
        seeing = pd.read_sql(query, conn)['seeingFwhmEff'].values
        frac_seeing[run][i] = len(np.where(seeing < 1.2)[0]) / len(seeing)
        query = f"select observationStartMJD, night, filter, moonPhase from observations where scheduler_note like '%{i}%'"
        vals = pd.read_sql(query, conn)
        # Find times with 60 u band visits in 30 nights
        counts = vals.query("filter == 'u'").groupby('night').count()['observationStartMJD']
        visits_per_30, bin_edges, bin_count = binned_statistic(counts.index, counts.values, bins=np.arange(nstart, 3652, 30), statistic='sum')
        max_u_per_30[run][i] = max(visits_per_30)
        u_per_30[run][i] = len(np.where(visits_per_30 > 60)[0])
        # Find pairs of nights with the required number of visits per pair-night
        t = vals.groupby(['night', 'filter']).agg({'filter': 'count'}).rename(columns={'filter': 'count'}).reset_index().pivot(index=['night'], columns=['filter']).fillna(0)
        t = t.droplevel(level=0, axis=1)
        t = t.drop(labels='u', axis=1)
        t['total'] = t.sum(axis=1)
        t = t.query('total > 0')
        s = t.query('g>1 and r>1 and i>3 and z>5 and y>4')
        # find and drop all the impossible y combinations
        def find_drops(df, column, minimum_val):
            idx_bad = np.where(df[column] < minimum_val)[0]
            idx_good = np.where(df[column] >= minimum_val)[0]
            drop = []
            for i in idx_bad:
                if (i+1 not in idx_good) and (i-1 not in idx_good):
                    drop.append(i)
            return drop
        drop = find_drops(t, 'y', 4)
        t = t.drop(index=t.iloc[drop].index)
        drop = find_drops(t, 'z', 5)
        t = t.drop(index=t.iloc[drop].index)
        drop = find_drops(t, 'i', 3)
        t = t.drop(index=t.iloc[drop].index)
        drop = find_drops(t, 'r', 1)
        t = t.drop(index=t.iloc[drop].index)
        drop = find_drops(t, 'g', 1)
        t = t.drop(index=t.iloc[drop].index)
        nights = t.index.values
        pairs = np.where(np.diff(nights) < 2)[0]
        pnights = t.iloc[np.sort(np.concatenate([pairs, pairs+1]))].index.values
        double_count = [p for p in pnights if p in s.index.values]
        pair_count[run][i] = len(pairs) + len(s) - len(double_count)
        
pdfs = {}
for i in ddfs:
    ra = ddfs[i].ra.deg
    dec = ddfs[i].dec.deg
    coord = ddfs[i]
    ra_hours = coord.ra.hms
    eclip_lat = coord.barycentrictrueecliptic.lat.deg
    eclip_lon = coord.barycentrictrueecliptic.lon.deg
    gal_lon = coord.galactic.l.deg
    gal_lat = coord.galactic.b.deg
    pdfs[i] = [ra, dec, gal_lon, gal_lat, eclip_lon, eclip_lat]
d = pd.DataFrame(pdfs, index=['RA', 'Dec', 'Gal l', 'Gal b', 'Eclip l', 'Eclip b']).round(2)
nvisits = pd.DataFrame(nvis).T
seeing = pd.DataFrame(frac_seeing).T
max_u_per_30 = pd.DataFrame(max_u_per_30).T
u_per_30 = pd.DataFrame(u_per_30).T
night_pairs = pd.DataFrame(pair_count).T

In [20]:
display(Markdown("Basic DDF information"))
display(d)

display(Markdown("Number of visits per DDF"))
display(nvisits)

display(Markdown("Fraction of DDF r-band visits with seeing < 1.2\""))
display(seeing.round(2))

display(Markdown("Maximum number of u band visits within 30 nights"))
display(max_u_per_30)

display(Markdown("Number of months with >60 u band visits in 30 nights"))
display(u_per_30)

display(Markdown("Number of 48-hour intervals with g>1, r>1, i>3, z>5, y>4 visits"))
display(night_pairs)

Basic DDF information

,ELAISS1,XMM_LSS,ECDFS,COSMOS,EDFS_a,EDFS_b
RA,9.45,35.57,52.98,150.11,58.90,63.60
Dec,-44.02,-4.82,-28.12,2.23,-49.32,-47.60
Gal l,311.29,171.10,224.07,236.78,257.90,254.48
Gal b,-72.88,-58.91,-54.60,42.13,-48.46,-45.77
Eclip l,346.66,31.59,40.81,151.39,32.00,40.97
Eclip b,-43.20,-17.92,-45.44,-9.34,-66.61,-66.60


Number of visits per DDF

,ELAISS1,XMM_LSS,ECDFS,COSMOS,EDFS_a,EDFS_b
baseline_v3.4_10yrs,20768,22222,22217,42240,11518,11483
ddf_acor_sf10_lsf15_lsr10_v3.4_10yrs,20998,20852,21672,43080,11347,11324
ddf_acor_sf10_lsf15_lsr20_v3.4_10yrs,20509,20490,21632,42265,11300,11286
ddf_acor_sf10_lsf15_lsr50_v3.4_10yrs,19903,19924,21712,42608,11288,11269
ddf_acor_sf10_lsf20_lsr10_v3.4_10yrs,21289,21919,22685,43529,11211,11204
ddf_acor_sf10_lsf20_lsr20_v3.4_10yrs,21281,21984,22318,42933,11558,11515
ddf_acor_sf10_lsf20_lsr50_v3.4_10yrs,20896,20977,21383,42992,11407,11382
ddf_acor_sf10_lsf30_lsr10_v3.4_10yrs,21843,22283,22603,45764,11233,11207
ddf_acor_sf10_lsf30_lsr20_v3.4_10yrs,21414,22355,21902,45498,10929,10917
ddf_acor_sf10_lsf30_lsr50_v3.4_10yrs,20640,21295,21689,43512,11194,11176


Fraction of DDF r-band visits with seeing < 1.2"

...

,ELAISS1,XMM_LSS,ECDFS,COSMOS,EDFS_a,EDFS_b
baseline_v3.4_10yrs,24.0,32.0,16.0,112.0,12.0,12.0
ddf_acor_sf10_lsf15_lsr10_v3.4_10yrs,16.0,24.0,16.0,120.0,12.0,12.0
ddf_acor_sf10_lsf15_lsr20_v3.4_10yrs,16.0,24.0,16.0,120.0,12.0,12.0
ddf_acor_sf10_lsf15_lsr50_v3.4_10yrs,16.0,24.0,16.0,120.0,12.0,12.0
ddf_acor_sf10_lsf20_lsr10_v3.4_10yrs,17.0,16.0,16.0,120.0,8.0,8.0
ddf_acor_sf10_lsf20_lsr20_v3.4_10yrs,16.0,16.0,16.0,120.0,12.0,12.0
ddf_acor_sf10_lsf20_lsr50_v3.4_10yrs,16.0,24.0,16.0,120.0,12.0,12.0
ddf_acor_sf10_lsf30_lsr10_v3.4_10yrs,24.0,24.0,24.0,120.0,12.0,12.0
ddf_acor_sf10_lsf30_lsr20_v3.4_10yrs,32.0,24.0,24.0,120.0,12.0,12.0
ddf_acor_sf10_lsf30_lsr50_v3.4_10yrs,17.0,24.0,16.0,120.0,12.0,12.0


Number of months with >60 u band visits in 30 nights

,ELAISS1,XMM_LSS,ECDFS,COSMOS,EDFS_a,EDFS_b
baseline_v3.4_10yrs,0,0,0,13,0,0
ddf_acor_sf10_lsf15_lsr10_v3.4_10yrs,0,0,0,12,0,0
ddf_acor_sf10_lsf15_lsr20_v3.4_10yrs,0,0,0,12,0,0
ddf_acor_sf10_lsf15_lsr50_v3.4_10yrs,0,0,0,13,0,0
ddf_acor_sf10_lsf20_lsr10_v3.4_10yrs,0,0,0,13,0,0
ddf_acor_sf10_lsf20_lsr20_v3.4_10yrs,0,0,0,14,0,0
ddf_acor_sf10_lsf20_lsr50_v3.4_10yrs,0,0,0,13,0,0
ddf_acor_sf10_lsf30_lsr10_v3.4_10yrs,0,0,0,13,0,0
ddf_acor_sf10_lsf30_lsr20_v3.4_10yrs,0,0,0,14,0,0
ddf_acor_sf10_lsf30_lsr50_v3.4_10yrs,0,0,0,12,0,0


Number of 48-hour intervals with g>1, r>1, i>3, z>5, y>4 visits

,ELAISS1,XMM_LSS,ECDFS,COSMOS,EDFS_a,EDFS_b
baseline_v3.4_10yrs,138,142,163,154,164,164
ddf_acor_sf10_lsf15_lsr10_v3.4_10yrs,141,128,151,148,158,160
ddf_acor_sf10_lsf15_lsr20_v3.4_10yrs,136,125,146,142,165,166
ddf_acor_sf10_lsf15_lsr50_v3.4_10yrs,136,129,154,150,155,155
ddf_acor_sf10_lsf20_lsr10_v3.4_10yrs,146,146,159,152,160,161
ddf_acor_sf10_lsf20_lsr20_v3.4_10yrs,142,143,159,144,165,165
ddf_acor_sf10_lsf20_lsr50_v3.4_10yrs,137,131,150,148,159,160
ddf_acor_sf10_lsf30_lsr10_v3.4_10yrs,146,138,150,157,154,154
ddf_acor_sf10_lsf30_lsr20_v3.4_10yrs,147,140,159,149,153,155
ddf_acor_sf10_lsf30_lsr50_v3.4_10yrs,140,134,155,152,161,161


In [21]:
for i in ddfs:
    if i == 'EDFS_a':
        name = 'EDFS'
    elif i == "EDFS_b":
        continue
    else:
        name = i
    msub = metric_subsets.loc['DDF Depths'].query("metric.str.contains(@name) and metric.str.contains('Coadd')")
    print(name)
    display(summaries.loc[all_runs, msub['metric']].round(2).rename(columns=msub['short_name']))

ELAISS1


metric,CoaddedM5 ELAISS1 u,CoaddedM5 ELAISS1 g,CoaddedM5 ELAISS1 r,CoaddedM5 ELAISS1 i,CoaddedM5 ELAISS1 z,CoaddedM5 ELAISS1 y
run,,,,,,
baseline_v3.4_10yrs,26.54,28.27,28.25,27.87,27.36,25.97
ddf_acor_sf10_lsf15_lsr10_v3.4_10yrs,26.51,28.28,28.27,27.89,27.36,26.04
ddf_acor_sf10_lsf15_lsr20_v3.4_10yrs,26.51,28.22,28.25,27.85,27.35,26.02
ddf_acor_sf10_lsf15_lsr50_v3.4_10yrs,26.48,28.20,28.22,27.82,27.30,25.98
ddf_acor_sf10_lsf20_lsr10_v3.4_10yrs,26.62,28.26,28.30,27.90,27.39,26.03
ddf_acor_sf10_lsf20_lsr20_v3.4_10yrs,26.60,28.29,28.29,27.90,27.39,26.02
ddf_acor_sf10_lsf20_lsr50_v3.4_10yrs,26.53,28.26,28.26,27.87,27.36,26.01
ddf_acor_sf10_lsf30_lsr10_v3.4_10yrs,26.64,28.35,28.35,27.97,27.49,26.15
ddf_acor_sf10_lsf30_lsr20_v3.4_10yrs,26.61,28.34,28.34,27.94,27.44,26.11


XMM_LSS


metric,CoaddedM5 XMM_LSS u,CoaddedM5 XMM_LSS g,CoaddedM5 XMM_LSS r,CoaddedM5 XMM_LSS i,CoaddedM5 XMM_LSS z,CoaddedM5 XMM_LSS y
run,,,,,,
baseline_v3.4_10yrs,26.52,28.17,28.20,27.81,27.30,25.96
ddf_acor_sf10_lsf15_lsr10_v3.4_10yrs,26.45,28.12,28.13,27.75,27.27,25.93
ddf_acor_sf10_lsf15_lsr20_v3.4_10yrs,26.47,28.14,28.13,27.76,27.24,25.88
ddf_acor_sf10_lsf15_lsr50_v3.4_10yrs,26.43,28.09,28.09,27.72,27.23,25.90
ddf_acor_sf10_lsf20_lsr10_v3.4_10yrs,26.46,28.20,28.17,27.82,27.29,26.00
ddf_acor_sf10_lsf20_lsr20_v3.4_10yrs,26.46,28.20,28.19,27.82,27.32,26.00
ddf_acor_sf10_lsf20_lsr50_v3.4_10yrs,26.44,28.13,28.13,27.78,27.28,25.94
ddf_acor_sf10_lsf30_lsr10_v3.4_10yrs,26.59,28.28,28.27,27.91,27.40,26.09
ddf_acor_sf10_lsf30_lsr20_v3.4_10yrs,26.56,28.24,28.26,27.89,27.38,26.06


ECDFS


metric,CoaddedM5 ECDFS u,CoaddedM5 ECDFS g,CoaddedM5 ECDFS r,CoaddedM5 ECDFS i,CoaddedM5 ECDFS z,CoaddedM5 ECDFS y
run,,,,,,
baseline_v3.4_10yrs,26.61,28.34,28.34,27.95,27.43,26.14
ddf_acor_sf10_lsf15_lsr10_v3.4_10yrs,26.58,28.24,28.26,27.87,27.38,26.04
ddf_acor_sf10_lsf15_lsr20_v3.4_10yrs,26.63,28.28,28.28,27.90,27.38,26.01
ddf_acor_sf10_lsf15_lsr50_v3.4_10yrs,26.56,28.29,28.27,27.89,27.37,26.07
ddf_acor_sf10_lsf20_lsr10_v3.4_10yrs,26.63,28.35,28.34,27.96,27.44,26.10
ddf_acor_sf10_lsf20_lsr20_v3.4_10yrs,26.67,28.36,28.33,27.94,27.42,26.08
ddf_acor_sf10_lsf20_lsr50_v3.4_10yrs,26.59,28.25,28.27,27.89,27.39,26.06
ddf_acor_sf10_lsf30_lsr10_v3.4_10yrs,26.75,28.44,28.44,28.05,27.54,26.17
ddf_acor_sf10_lsf30_lsr20_v3.4_10yrs,26.69,28.40,28.38,28.01,27.46,26.15


COSMOS


metric,CoaddedM5 COSMOS u,CoaddedM5 COSMOS g,CoaddedM5 COSMOS r,CoaddedM5 COSMOS i,CoaddedM5 COSMOS z,CoaddedM5 COSMOS y
run,,,,,,
baseline_v3.4_10yrs,26.92,28.55,28.55,28.17,27.67,26.24
ddf_acor_sf10_lsf15_lsr10_v3.4_10yrs,26.94,28.57,28.58,28.20,27.70,26.25
ddf_acor_sf10_lsf15_lsr20_v3.4_10yrs,26.93,28.56,28.56,28.18,27.68,26.23
ddf_acor_sf10_lsf15_lsr50_v3.4_10yrs,26.92,28.55,28.57,28.18,27.68,26.23
ddf_acor_sf10_lsf20_lsr10_v3.4_10yrs,26.96,28.59,28.59,28.22,27.72,26.26
ddf_acor_sf10_lsf20_lsr20_v3.4_10yrs,26.95,28.58,28.58,28.21,27.71,26.25
ddf_acor_sf10_lsf20_lsr50_v3.4_10yrs,26.94,28.57,28.58,28.20,27.69,26.24
ddf_acor_sf10_lsf30_lsr10_v3.4_10yrs,27.00,28.61,28.62,28.24,27.75,26.29
ddf_acor_sf10_lsf30_lsr20_v3.4_10yrs,26.98,28.62,28.62,28.23,27.75,26.29


EDFS


metric,CoaddedM5 EDFS u,CoaddedM5 EDFS g,CoaddedM5 EDFS r,CoaddedM5 EDFS i,CoaddedM5 EDFS z,CoaddedM5 EDFS y
run,,,,,,
baseline_v3.4_10yrs,26.38,28.10,28.10,27.70,27.18,25.86
ddf_acor_sf10_lsf15_lsr10_v3.4_10yrs,26.38,28.09,28.13,27.71,27.20,25.89
ddf_acor_sf10_lsf15_lsr20_v3.4_10yrs,26.38,28.09,28.11,27.70,27.18,25.90
ddf_acor_sf10_lsf15_lsr50_v3.4_10yrs,26.42,28.09,28.09,27.67,27.16,25.81
ddf_acor_sf10_lsf20_lsr10_v3.4_10yrs,26.39,28.10,28.10,27.72,27.19,25.89
ddf_acor_sf10_lsf20_lsr20_v3.4_10yrs,26.40,28.10,28.10,27.72,27.22,25.90
ddf_acor_sf10_lsf20_lsr50_v3.4_10yrs,26.37,28.11,28.14,27.71,27.19,25.88
ddf_acor_sf10_lsf30_lsr10_v3.4_10yrs,26.50,28.18,28.16,27.79,27.28,25.93
ddf_acor_sf10_lsf30_lsr20_v3.4_10yrs,26.41,28.08,28.09,27.71,27.18,25.89
